In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.chdir('/content/drive/MyDrive')

!mkdir -p olist-customer-intelligence/{data/raw,data/processed,notebooks,sql,src,models,reports,app/pages,docs}
os.chdir('/content/drive/MyDrive/M5-forecasting')
!touch src/__init__.py
print("Now in:", os.getcwd())

Now in: /content/drive/MyDrive/M5-forecasting


In [3]:
%%writefile src/models/naive_baseline.py
"""Seasonal naive baseline: forecast = value from 7 days ago (weekly season)."""
import pandas as pd
import numpy as np


def seasonal_naive_forecast(series, horizon=28, season=7):
    """series: 1D array of historical daily sales (sorted by date).
    Returns horizon-length forecast by tiling the last `season` values."""
    last_season = np.asarray(series)[-season:]
    reps = int(np.ceil(horizon / season))
    return np.tile(last_season, reps)[:horizon]


def forecast_all(agg_df, horizon=28, season=7, key="series_id"):
    """agg_df: long frame with columns [key, date, sales], sorted.
    Returns DataFrame [key, date, yhat] for the horizon after each series' last date."""
    out = []
    for sid, g in agg_df.groupby(key, observed=True):
        g = g.sort_values("date")
        yhat = seasonal_naive_forecast(g["sales"].values, horizon, season)
        future_dates = pd.date_range(g["date"].max() + pd.Timedelta(days=1),
                                     periods=horizon, freq="D")
        out.append(pd.DataFrame({key: sid, "date": future_dates, "yhat": yhat}))
    return pd.concat(out, ignore_index=True)

Overwriting src/models/naive_baseline.py


In [4]:
%%writefile src/models/prophet_model.py
"""Prophet forecasting at aggregated level with events & holidays."""
import pandas as pd
import numpy as np
from prophet import Prophet


def build_holidays(cal_events):
    """cal_events: DataFrame [date, event_name] of non-null event days.
    Returns Prophet-format holidays frame."""
    h = cal_events.rename(columns={"date": "ds", "event_name": "holiday"})
    h = h[["holiday", "ds"]].drop_duplicates()
    # Christmas closures as their own holiday
    xmas = pd.DataFrame({
        "holiday": "Christmas",
        "ds": pd.to_datetime(["2014-12-25", "2015-12-25"]),
    })
    return pd.concat([h, xmas], ignore_index=True)


def fit_predict_one(train_df, holidays, horizon=28, snap_future=None):
    """train_df: [ds, y, snap]. Returns forecast frame incl. yhat, lower, upper."""
    m = Prophet(
        holidays=holidays,
        weekly_seasonality=True,
        yearly_seasonality=True,
        daily_seasonality=False,
        seasonality_mode="multiplicative",
        interval_width=0.80,   # gives ~P10/P90 band
    )
    m.add_regressor("snap")
    m.fit(train_df)
    future = m.make_future_dataframe(periods=horizon, freq="D")
    # attach snap for future rows
    future = future.merge(train_df[["ds", "snap"]], on="ds", how="left")
    if snap_future is not None:
        future = future.merge(snap_future, on="ds", how="left", suffixes=("", "_f"))
        future["snap"] = future["snap"].fillna(future["snap_f"])
    future["snap"] = future["snap"].fillna(0)
    fcst = m.predict(future)
    return m, fcst

Overwriting src/models/prophet_model.py


In [5]:
import sys; sys.path.append("..")
import pandas as pd, numpy as np
import pickle
from src.models.naive_baseline import forecast_all
from src.models.prophet_model import build_holidays, fit_predict_one

In [6]:
feat = pd.read_parquet("data/processed/features.parquet",
                       columns=["id","store_id","cat_id","state_id","date","sales",
                                "snap","event_name_1"])

feat["series_id"] = feat["store_id"].astype(str) + "__" + feat["cat_id"].astype(str)
agg = (feat.groupby(["series_id","date"], observed=True)
           .agg(sales=("sales","sum"), snap=("snap","max"))
           .reset_index())
print(agg["series_id"].nunique(), "series")   # expect 30

cutoff = agg["date"].max() - pd.Timedelta(days=28)
train = agg[agg["date"] <= cutoff]
test  = agg[agg["date"] >  cutoff]
print("train through", cutoff.date(), "| test days:", test["date"].nunique())

30 series
train through 2016-04-24 | test days: 28


In [7]:
agg.shape

(21060, 4)

In [8]:
naive_fc = forecast_all(train.rename(columns={"series_id":"series_id"}),
                        horizon=28, key="series_id")
naive_eval = naive_fc.merge(test, on=["series_id","date"], how="inner")
naive_rmse = np.sqrt(((naive_eval["yhat"] - naive_eval["sales"])**2).mean())
print("Seasonal naive RMSE:", round(naive_rmse, 1))

Seasonal naive RMSE: 308.1


In [9]:
events = feat[feat["event_name_1"].notna()][["date","event_name_1"]].drop_duplicates()
events = events.rename(columns={"event_name_1":"event_name"})
holidays = build_holidays(events)
print(holidays.shape, "holiday rows")

(58, 2) holiday rows


In [10]:
!pip install prophet -q

In [11]:
prophet_models = {}
prophet_rows = []

for sid, g in train.groupby("series_id", observed=True):
    tr = g.rename(columns={"date":"ds","sales":"y"})[["ds","y","snap"]].sort_values("ds")
    snap_future = test[test.series_id==sid].rename(columns={"date":"ds"})[["ds","snap"]]
    m, fcst = fit_predict_one(tr, holidays, horizon=28, snap_future=snap_future)
    prophet_models[sid] = m
    tail = fcst.tail(28)[["ds","yhat","yhat_lower","yhat_upper"]].copy()
    tail["series_id"] = sid
    prophet_rows.append(tail)

prophet_fc = pd.concat(prophet_rows, ignore_index=True).rename(columns={"ds":"date"})
print(prophet_fc.shape)

(840, 5)


In [12]:
prophet_eval = prophet_fc.merge(test, on=["series_id","date"], how="inner")
prophet_rmse = np.sqrt(((prophet_eval["yhat"] - prophet_eval["sales"])**2).mean())
print("Prophet RMSE:", round(prophet_rmse,1), "| Naive RMSE:", round(naive_rmse,1))
print("Prophet beats naive:", prophet_rmse < naive_rmse)

Prophet RMSE: 209.8 | Naive RMSE: 308.1
Prophet beats naive: True


In [14]:
prophet_fc.to_parquet("data/processed/forecasts_prophet.parquet", index=False)
with open("models/prophet_models.pkl","wb") as f:
    pickle.dump(prophet_models, f)
print("saved")

saved


## Naive Baseline + Prophet

**Models built:**
- **Seasonal naive** (`src/models/naive_baseline.py`): forecast = value 7 days prior, tiled over 28-day horizon. The M5 benchmark denominator.
- **Prophet** (`src/models/prophet_model.py`): fit per series at **store × category = 30 series**, with US holidays, Christmas-closure regressor, SNAP regressor, weekly + yearly multiplicative seasonality, 80% interval (~P10/P90).

**Setup:**
- Held out last 28 days as test; trained on the rest
- Aggregated to 30 smooth series (statistical models' hierarchical role; leaves handled by ML/deep)

**Results (28-day test, RMSE):**
- Seasonal naive: **308.1**
- Prophet: **209.8** → **beats naive by ~32%**

**Interpretation (honest framing):**
- This is Prophet's strength — smooth aggregates. Expected win, not the project headline.
- Hard part is the 30,490 intermittent leaves, where naive is far tougher to beat → that's LightGBM/deep's job.
- Final verdict comes from **WRMSSE across all hierarchy levels** (Day 25), not aggregate RMSE.

**Saved:** `forecasts_prophet.parquet`, `models/prophet_models.pkl`